In [3]:
import torch

In [5]:
inputs = torch.tensor([
    [0.43,0.15,0.89],   #Your(x1)
    [0.55,0.87,0.66],   #journey(x2)
    [0.57,0.85,0.64],   #starts(x3)
    [0.22,0.58,0.33],   #with(x4)
    [0.77,0.25,0.10],   #one(x5)
    [0.05,0.80,0.55]    #step(x6)
])

#Assume 3dimensional embeddings 

In [7]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2 #output_embedding shape

#Usually input and output dimensions are same

In [8]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

#for now we are turning off the gradients..but when we will be training we will set them to `True`

In [10]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


In [11]:
keys = inputs @ W_key
values = inputs @ W_value
print(f"keys.shape -> {keys.shape}")
print(f"values.shape -> {values.shape}")

keys.shape -> torch.Size([6, 2])
values.shape -> torch.Size([6, 2])


In [12]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [13]:
attn_score_2 = query_2 @ keys.T
print(attn_score_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [14]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_score_2 / d_k ** 0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [15]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


### Self-Attention

In [16]:
#class Self-Attention

import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
        
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim = -1
        )
        context_vec = attn_weights @ values
        return context_vec
        
        

In [17]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [18]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim = -1
        )
        context_vec = attn_weights @ values
        return context_vec

* We can improve V1 by utilizing PyTorch's `nn.Linear` layers, which effectively perform matrix multiplication when the bias are disabled.
* Another advantage is `nn.Linear` has an optimized weight initialization scheme, contributing to more stable and effective training

In [23]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[ 0.0293, -0.3242],
        [ 0.0284, -0.3250],
        [ 0.0284, -0.3251],
        [ 0.0271, -0.3272],
        [ 0.0267, -0.3278],
        [ 0.0277, -0.3262]], grad_fn=<MmBackward0>)


#### Casual Attention/Masked Attention


In [24]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1667, 0.1635, 0.1641, 0.1672, 0.1767, 0.1618],
        [0.1675, 0.1622, 0.1626, 0.1688, 0.1746, 0.1642],
        [0.1675, 0.1623, 0.1627, 0.1687, 0.1745, 0.1643],
        [0.1672, 0.1642, 0.1644, 0.1680, 0.1705, 0.1657],
        [0.1672, 0.1646, 0.1648, 0.1678, 0.1694, 0.1661],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
context_length = attn_scores.shape[0] #no of tokens in i/p seq
mask_simple = torch.tril(torch.ones(context_length,  context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


`torch.tril(...)`<br>
Purpose: Applies a lower-triangular filter to the grid of ones.<br>
How it works: tril stands for "triangular lower". `It keeps all the numbers on the main diagonal and below them exactly as they are. Every number located above the diagonal is wiped out and changed to 0.0.`

In [27]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1667, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1675, 0.1622, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1675, 0.1623, 0.1627, 0.0000, 0.0000, 0.0000],
        [0.1672, 0.1642, 0.1644, 0.1680, 0.0000, 0.0000],
        [0.1672, 0.1646, 0.1648, 0.1678, 0.1694, 0.0000],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<MulBackward0>)


In [28]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5080, 0.4920, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3401, 0.3295, 0.3303, 0.0000, 0.0000, 0.0000],
        [0.2519, 0.2474, 0.2477, 0.2530, 0.0000, 0.0000],
        [0.2005, 0.1975, 0.1976, 0.2012, 0.2032, 0.0000],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<DivBackward0>)


##### Efficient workflow of casual attention is:
* Start<br>
`Take unnormalised attention scores` 
* mask with -inf above diagonal<br>
`Now we get masked attention scores which are unnormalised`
* apply softmax<br>
`Normalized Masked Attention weights`

In [29]:
mask = torch.triu(torch.ones(context_length,context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[-0.0634,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0921, -0.1375,    -inf,    -inf,    -inf,    -inf],
        [-0.0908, -0.1357, -0.1322,    -inf,    -inf,    -inf],
        [-0.0514, -0.0774, -0.0757, -0.0452,    -inf,    -inf],
        [-0.0420, -0.0635, -0.0623, -0.0367, -0.0229,    -inf],
        [-0.0672, -0.1005, -0.0980, -0.0596, -0.0254, -0.0873]],
       grad_fn=<MaskedFillBackward0>)


* This code first creates a upper triangular matrix with 1 above diagonal
* Then we convert the mask into boolenan value and replace the value which is True ->`-inf`


In [31]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5080, 0.4920, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3401, 0.3295, 0.3303, 0.0000, 0.0000, 0.0000],
        [0.2519, 0.2474, 0.2477, 0.2530, 0.0000, 0.0000],
        [0.2005, 0.1975, 0.1976, 0.2012, 0.2032, 0.0000],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<SoftmaxBackward0>)


#### Masking additional attention weights with dropout
* it prevents overfitting
* In GPT like models, dropout is applied twice generally:
* After calculating attention weights to the value vectors
* but we will apply dropout mask after computing attention weights, because its more common in practice 

In [32]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


`When applying dropout to an attention weight matrix with a rate of 50%, half of the elements in the matrix are randomly set to zero. To compensate for the reduction in active elements, the values of the remaining elements are scaled up by a factor of 1/0.5 = 2. This scaling is crucial to maintain the overall balance of the attention weights `

In [33]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6803, 0.6590, 0.6607, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4947, 0.4953, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3949, 0.0000, 0.4025, 0.0000, 0.0000],
        [0.0000, 0.3269, 0.3274, 0.3365, 0.3447, 0.0000]],
       grad_fn=<MulBackward0>)


In [35]:
#making a batch to test

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [38]:
class CasualAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.transpose(1,2) #no need to transpose batch dim
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** -0.5, dim = -1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

*In PyTorch, `self.register_buffer()` is an nn.Module method used to save state variables that are essential to your model's operations but are not learnable parameters updated by gradients

In [39]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CasualAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print(f"context_vecs.shape -> {context_vecs.shape}")

context_vecs.shape -> torch.Size([2, 6, 2])
